In [1]:
import pandas as pd
import pickle
import redis

r = redis.Redis(host='localhost', port=6379, db=0)

DATA_DIR = "../../../Recommender/data/output"

In [2]:
users = pd.read_parquet(f"{DATA_DIR}/user.parquet")

filtered_user_ids = users["user_id"].unique()[:5]

filtered_users = users[
    users["user_id"].isin(filtered_user_ids)
]

filtered_users

,user_id,num_reviews,avg_rating_given,rating_std,min_rating_given,max_rating_given,first_review_date,last_review_date,total_recency_weight,total_helpful_votes_received,...,extreme_rating_ratio,positive_rating_ratio,negative_rating_ratio,avg_price_purchased,free_app_ratio,days_active,reviews_per_day,verified_purchase_ratio,user_segment,is_discriminating
0,ae2222frpdmnomyomcwiantxp7uq,1,5.000000,NaN,5.0,5.0,2014-07-06,2014-07-06,0.000103,0,...,1.000000,1.000000,0.000000,-1.000000,0.000000,1,1.000000,1.0,one_time,False
1,ae22232ob6s7uac75jufyrgbshiq,1,5.000000,NaN,5.0,5.0,2020-05-16,2020-05-16,0.036116,1,...,1.000000,1.000000,0.000000,0.000000,1.000000,1,1.000000,1.0,one_time,False
2,ae22236afrrsmqikgg7tptb75qea,9,3.444444,1.878238,1.0,5.0,2012-11-22,2018-12-15,0.009146,79,...,0.777778,0.666667,0.333333,0.552222,0.333333,2215,0.004063,1.0,occasional,True
3,ae2224fsuk5av5r2usyxinuntw7q,3,2.333333,2.309401,1.0,5.0,2015-03-10,2015-03-24,0.000614,0,...,1.000000,0.333333,0.666667,0.000000,1.000000,15,0.200000,1.0,occasional,True
4,ae2226penzttcdkfgrtucux2nu2q,2,3.000000,2.828427,1.0,5.0,2021-05-13,2021-05-13,0.194609,3,...,1.000000,0.500000,0.500000,0.000000,1.000000,1,2.000000,1.0,occasional,True


In [3]:
reviews = pd.read_parquet(f"{DATA_DIR}/user-item-interaction.parquet")

filtered_user_reviews = reviews[reviews["user_id"].isin(filtered_user_ids)]

filtered_user_reviews

,review_id,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,helpful_vote,...,num_item_img,num_item_videos,is_free,has_price_info,price_bucket,num_categories,days_since_review,review_year,review_month,recency_weight
346794,346794,b0117u0g3m,b0117u0g3m,ae22236afrrsmqikgg7tptb75qea,2018-12-15,what is going on!!!,it was awesome i played so much but then i did...,[],True,44,...,6,1,True,True,unknown,0,1731,2018,12,0.008745
346795,346795,b00dugcly4,b00dugcly4,ae22236afrrsmqikgg7tptb75qea,2014-11-09,good,fun easy nice to use for thinking,[],True,0,...,6,1,True,True,unknown,0,3228,2014,11,0.000145
346796,346796,b00dzyxtry,b00dzyxtry,ae22236afrrsmqikgg7tptb75qea,2013-08-13,we all love playing it,everyone in my family loves this game. even m...,[],True,23,...,6,1,False,True,1-10,0,3681,2013,8,0.000042
346797,346797,b00b2v66vs,b00b2v66vs,ae22236afrrsmqikgg7tptb75qea,2013-07-29,game has a flaw,we loved playing this game. it does sometimes...,[],True,2,...,6,1,False,False,NaN,0,3696,2013,7,0.000040
346798,346798,b00bqpeit2,b00bqpeit2,ae22236afrrsmqikgg7tptb75qea,2013-07-29,game has a serious flaw,"i really loved this game. i played it, the ki...",[],True,2,...,6,0,False,False,NaN,0,3696,2013,7,0.000040
346799,346799,b00d948ss0,b00d948ss0,ae22236afrrsmqikgg7tptb75qea,2013-07-12,nice word search game,nice game. easy to play and understand. the ...,[],True,1,...,6,1,True,True,unknown,0,3713,2013,7,0.000038
346800,346800,b00drpve3m,b00drpve3m,ae22236afrrsmqikgg7tptb75qea,2013-07-12,fun to play,"who doesn't love uno? i like it in real life,...",[],True,5,...,6,1,False,False,NaN,0,3713,2013,7,0.000038
346801,346801,b00b7s5fdg,b00b7s5fdg,ae22236afrrsmqikgg7tptb75qea,2013-06-08,my 5 year old loves it,my youngest loves this app. she likes that sh...,[],True,2,...,12,0,False,True,1-10,0,3747,2013,6,0.000035
346802,346802,b006cq8tc2,b006cq8tc2,ae22236afrrsmqikgg7tptb75qea,2012-11-22,the boys love it,the adult boys are playing away with this game...,[],True,0,...,6,1,False,True,0-1,0,3945,2012,11,0.000020
597558,597558,b002zf31nq,b002zf31nq,ae2222frpdmnomyomcwiantxp7uq,2014-07-06,absolutely terrific. learning a language is tough,absolutely terrific. learning a language is t...,[],True,0,...,27,1,False,False,NaN,3,3354,2014,7,0.000103


In [4]:
items = pd.read_parquet(f"{DATA_DIR}/item.parquet")

filtered_items = items[items["parent_asin"].isin(filtered_user_reviews["parent_asin"])]

filtered_items

,parent_asin,num_reviews,avg_rating,rating_std,min_rating,max_rating,first_review_date,last_review_date,total_recency_weight,total_helpful_votes,...,num_item_img,num_item_videos,num_categories,is_free,price_bucket,days_on_platform,reviews_per_day,verified_review_ratio,popularity_segment,quality_score
10074,b002zf31nq,318,4.600629,0.919950,1.0,5.0,2009-12-16,2023-02-26,1.249999,3971,...,27,1,3,False,NaN,4821,0.065961,0.723270,popular,0.861393
15979,b006cq8tc2,161,4.161491,1.060074,1.0,5.0,2011-11-23,2014-12-06,0.003745,138,...,6,1,0,False,0-1,1110,0.145045,0.975155,popular,0.726148
26548,b00b2v66vs,10811,4.338637,1.114834,1.0,5.0,2013-01-24,2023-06-29,21.980449,84225,...,6,1,0,False,NaN,3809,2.838278,0.984923,popular,0.824012
26833,b00b7s5fdg,280,4.432143,1.192435,1.0,5.0,2013-02-24,2021-08-24,0.611359,1133,...,12,0,0,False,1-10,3104,0.090206,0.950000,popular,0.815256
27297,b00bi60dmu,3987,2.997743,1.771788,1.0,5.0,2013-03-16,2022-12-05,123.302185,3340,...,11,0,0,True,unknown,3552,1.122466,0.986958,popular,0.439072
27726,b00bqpeit2,622,4.250804,1.223018,1.0,5.0,2013-03-09,2019-09-30,0.174814,12455,...,6,0,0,False,NaN,2397,0.259491,0.901929,popular,0.784402
30463,b00d948ss0,3431,3.738269,1.280826,1.0,5.0,2013-06-20,2023-01-04,9.170357,3408,...,6,1,0,True,unknown,3486,0.984223,0.994754,popular,0.631777
31221,b00drpve3m,6053,3.884355,1.343207,1.0,5.0,2013-07-09,2022-09-23,7.401763,14936,...,6,1,0,False,NaN,3364,1.799346,0.993061,popular,0.692154
31396,b00dugcly4,8893,4.422242,0.821268,1.0,5.0,2013-07-31,2023-06-07,3.733197,3736,...,6,1,0,True,unknown,3599,2.470964,0.998538,popular,0.875518
31673,b00dzyxtry,1589,4.654500,0.838059,1.0,5.0,2013-07-19,2023-05-03,5.548706,5485,...,6,1,0,False,1-10,3576,0.444351,0.955318,popular,0.915187


In [5]:
filtered_item_reviews = reviews[reviews["parent_asin"].isin(filtered_items["parent_asin"])]

filtered_item_reviews

,review_id,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,helpful_vote,...,num_item_img,num_item_videos,is_free,has_price_info,price_bucket,num_categories,days_since_review,review_year,review_month,recency_weight
79,79,b0117u0g3m,b0117u0g3m,ahfzunqfxsvvt6z6byke5clbx3kq,2022-06-19,love it,great video,[],True,19,...,6,1,True,True,unknown,0,449,2022,6,0.292499
123,123,b00b2v66vs,b00b2v66vs,ahfzunqfxsvvt6z6byke5clbx3kq,2017-06-16,one star,hate this game,[],True,0,...,6,1,False,False,NaN,0,2278,2017,6,0.001956
283,283,b00dugcly4,b00dugcly4,afejkj56od6h2xh55dghji2526ma,2015-05-19,fun,"sometimes easy, sometimes hard but definitely ...",[],True,0,...,6,1,True,True,unknown,0,3037,2015,5,0.000245
322,322,b00hgj18c2,b00hgj18c2,agdcd2znymdzavqkq3rz56ljtrvq,2018-04-02,zen of solitare,this is my go-to game when i need to quiet my ...,[],True,1,...,9,1,True,True,unknown,0,1988,2018,4,0.004327
378,378,b00ig2dokm,b00ig2dokm,agjut4z42gsgx2ygiyh574yntesq,2019-05-23,it's like all games that eventually you are as...,very fun game i believe it's better then farmv...,[],True,4,...,6,1,True,True,unknown,0,1572,2019,5,0.013516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4879932,4879932,b00qh8yuly,b00qh8yuly,aeh34tca74nm2pl63rbu5skixyga,2015-04-28,way to costly,way too costly for what you get. not enough ch...,[],True,3,...,7,1,True,True,unknown,0,3058,2015,4,0.000231
4879973,4879973,b0117u0g3m,b0117u0g3m,ah6khu2wtt3gzz2dyx3nym66n67a,2017-05-09,read this!!,you guys should delete profiles that have been...,[],True,10,...,6,1,True,True,unknown,0,2316,2017,5,0.001763
4880020,4880020,b00ig2dokm,b00ig2dokm,aeswzdo3yusjcx2xmc45p4hh7syq,2019-07-19,very slow updates,love this game. hate that the updates don't g...,[],True,129,...,6,1,True,True,unknown,0,1515,2019,7,0.015798
4880062,4880062,b00dugcly4,b00dugcly4,aexojavfrhflgl2nirjoflxfv2na,2015-06-25,amazing,the only thing is that they have an app for an...,[],True,0,...,6,1,True,True,unknown,0,3000,2015,6,0.000271


In [11]:
filtered_reviews = (
    pd.concat([filtered_user_reviews, filtered_item_reviews])
    .drop_duplicates(subset="review_id")
    .reset_index()
)

filtered_reviews

,index,review_id,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,...,num_item_img,num_item_videos,is_free,has_price_info,price_bucket,num_categories,days_since_review,review_year,review_month,recency_weight
0,346794,346794,b0117u0g3m,b0117u0g3m,ae22236afrrsmqikgg7tptb75qea,2018-12-15,what is going on!!!,it was awesome i played so much but then i did...,[],True,...,6,1,True,True,unknown,0,1731,2018,12,0.008745
1,346795,346795,b00dugcly4,b00dugcly4,ae22236afrrsmqikgg7tptb75qea,2014-11-09,good,fun easy nice to use for thinking,[],True,...,6,1,True,True,unknown,0,3228,2014,11,0.000145
2,346796,346796,b00dzyxtry,b00dzyxtry,ae22236afrrsmqikgg7tptb75qea,2013-08-13,we all love playing it,everyone in my family loves this game. even m...,[],True,...,6,1,False,True,1-10,0,3681,2013,8,0.000042
3,346797,346797,b00b2v66vs,b00b2v66vs,ae22236afrrsmqikgg7tptb75qea,2013-07-29,game has a flaw,we loved playing this game. it does sometimes...,[],True,...,6,1,False,False,NaN,0,3696,2013,7,0.000040
4,346798,346798,b00bqpeit2,b00bqpeit2,ae22236afrrsmqikgg7tptb75qea,2013-07-29,game has a serious flaw,"i really loved this game. i played it, the ki...",[],True,...,6,0,False,False,NaN,0,3696,2013,7,0.000040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93791,4879932,4879932,b00qh8yuly,b00qh8yuly,aeh34tca74nm2pl63rbu5skixyga,2015-04-28,way to costly,way too costly for what you get. not enough ch...,[],True,...,7,1,True,True,unknown,0,3058,2015,4,0.000231
93792,4879973,4879973,b0117u0g3m,b0117u0g3m,ah6khu2wtt3gzz2dyx3nym66n67a,2017-05-09,read this!!,you guys should delete profiles that have been...,[],True,...,6,1,True,True,unknown,0,2316,2017,5,0.001763
93793,4880020,4880020,b00ig2dokm,b00ig2dokm,aeswzdo3yusjcx2xmc45p4hh7syq,2019-07-19,very slow updates,love this game. hate that the updates don't g...,[],True,...,6,1,True,True,unknown,0,1515,2019,7,0.015798
93794,4880062,4880062,b00dugcly4,b00dugcly4,aexojavfrhflgl2nirjoflxfv2na,2015-06-25,amazing,the only thing is that they have an app for an...,[],True,...,6,1,True,True,unknown,0,3000,2015,6,0.000271


### Export Data

In [12]:
filtered_users.to_parquet("user.parquet")
filtered_items.to_parquet("item.parquet")
filtered_reviews.to_parquet("user-item-interaction.parquet")

### Cache into Redis


In [13]:
filtered_users = pd.read_parquet("user.parquet")
filtered_reviews = pd.read_parquet("user-item-interaction.parquet")
filtered_items = pd.read_parquet("item.parquet")

In [14]:
# Cache into Redis (no expiry)
r.set('users', pickle.dumps(filtered_users.to_dict('records')))
r.set('reviews', pickle.dumps(filtered_reviews.to_dict('records')))
r.set('items', pickle.dumps(filtered_items.to_dict('records')))

True